# Clinical trials corpus — load and publish

Loads `data/clinical_trials.parquet` (597,691 trials, 49 columns, 2.23 GiB on disk) as a
Hugging Face `Dataset` and pushes it to an existing dataset repo on the Hub.

**Before running anything:**

1. Install `datasets`. It is not in this venv, and neither is `pandas` any more — the
   outputs saved in the previous version of this notebook came from an earlier run, and
   its `import pandas` would fail today. `datasets` pulls `pandas` back in as a
   dependency. The corporate TLS proxy means `uv` needs `--native-tls`, or it dies with
   `invalid peer certificate: UnknownIssuer`:

   ```bash
   uv add --native-tls datasets
   ```

2. Export your token in the shell **before** starting the kernel:

   ```bash
   export HF_TOKEN=hf_xxxxxxxxxxxxxxxxxxxxx
   ```

   Putting it in `.env` is not reliable here — `load_dotenv` will not override a
   variable that is already set in the real environment.

3. Set `REPO_ID` in the config cell below, or export `HF_DATASET_REPO`.

## 1. Undo truststore's SSL injection

Run this **first**, before anything that opens a socket.

`pip_system_certs.pth` injects truststore into every process started from this venv — it
is not something you opt into — and on Python 3.14 its `wrap_socket` recurses until the C
stack is exhausted. Every outbound HTTPS call then dies with `RecursionError: Stack
overflow` raised deep inside `ssl.py`, which reads like a bug in your own code rather than
in the TLS layer. Stripping it falls back to certifi's CA bundle, which verifies the Hub
fine.

If the output still says `truststore`, restart the kernel and run this cell first — a
module that already captured the patched `SSLContext` keeps using it.

In [ ]:
import ssl
import sys

if sys.version_info >= (3, 14) and "truststore" in getattr(ssl.SSLContext, "__module__", ""):
    for _name in ("pip._vendor.truststore", "truststore"):
        try:
            __import__(_name, fromlist=["extract_from_ssl"]).extract_from_ssl()
        except Exception:  # not that copy of truststore; try the next
            continue
        if "truststore" not in ssl.SSLContext.__module__:
            print(f"removed truststore SSL injection ({_name})")
            break

print("python         :", sys.version.split()[0])
print("ssl.SSLContext :", ssl.SSLContext.__module__)

## 2. Configuration

`REPO_ID` **must** be changed from the placeholder. The push cell asserts on it, so a
stray Run All cannot create a junk repo.

In [ ]:
import os
from pathlib import Path

PARQUET = Path("../data/clinical_trials.parquet").resolve()

PLACEHOLDER = "YOUR-USER/YOUR-DATASET"
REPO_ID = os.environ.get("HF_DATASET_REPO", PLACEHOLDER)
SPLIT = "train"

# push_to_hub shards as it uploads. ~500 MB keeps each shard inside what the Dataset
# Viewer and streaming readers handle comfortably, and means a dropped transfer costs one
# shard rather than the whole 2.2 GB.
MAX_SHARD_SIZE = "500MB"

# datasets converts the parquet to Arrow on disk and memory-maps that, so this needs free
# DISK rather than free RAM -- budget several GB. Point it elsewhere if the default cache
# sits on a full drive.
CACHE_DIR = os.environ.get("HF_DATASETS_CACHE") or None

assert PARQUET.exists(), f"corpus not found: {PARQUET}"
print(f"parquet   : {PARQUET}")
print(f"size      : {PARQUET.stat().st_size / 1024 ** 3:.2f} GiB")
print(f"repo      : {REPO_ID}")
print(f"token set : {bool(os.environ.get('HF_TOKEN'))}")

## 3. Load the corpus

`Dataset.from_parquet` streams the file into an Arrow cache on disk and **memory-maps**
it, so resident memory stays flat no matter how large the corpus is.

That is the whole reason this notebook no longer uses pandas. The previous version's own
output reported `memory usage: 2.7+ GB` for the DataFrame, on a machine that often has
only ~2.7 GB free. Materialising this corpus in RAM is what starved the embedding model's
context allocation and produced `llama_decode returned -2`, and on another run `Failed to
create llama_context`. `pyarrow.parquet.read_table` has the same problem: parquet is
compressed on disk and is not compressed in memory.

In [ ]:
from datasets import Dataset

ds = Dataset.from_parquet(str(PARQUET), cache_dir=CACHE_DIR, split=SPLIT)
ds

## 4. Inspect

The Arrow schema is what the Hub will publish, so read it before pushing. Two things to
look at in this corpus specifically: the columns pandas reported as `object` —
`locations`, `outcomes`, `design_info`, the five date fields — land as nested Arrow
structs and lists, and `intervention_browse_module` is populated for only 273,137 of
597,691 rows. A column that arrives all-null is far cheaper to fix here than after a
2.2 GB upload.

In [ ]:
print(f"rows       : {ds.num_rows:,}")
print(f"columns    : {len(ds.column_names)}")
print(f"arrow bytes: {ds.data.nbytes / 1024 ** 3:.2f} GiB (memory-mapped, not resident)")
print()
for name, feature in ds.features.items():
    print(f"  {name:<28} {str(feature)[:90]}")

In [ ]:
import json

# Indexing one row decodes only that row, so this stays cheap on 597k rows.
for key, value in ds[12].items():
    text = json.dumps(value, default=str, ensure_ascii=False)
    print(f"{key:<28} {text[:150]}{'...' if len(text) > 150 else ''}")

## 5. Push to the Hub

`push_to_hub` does more than upload a file. It shards the corpus, writes the `README.md`
dataset card with the split sizes and config block, and lays the files out where the
Dataset Viewer expects them — none of which happens if you `hf upload` the raw parquet.

`hf_xet` is installed, so uploads are chunked and content-addressed: **if a shard fails,
re-run this cell** and the chunks already on the server are deduped rather than resent.
That matters here, because the proxy has a habit of killing long transfers.

If the Xet hosts are blocked — `transfer.xethub.hf.co` is a different domain from
`huggingface.co`, so the proxy may not allow it — set `HF_HUB_DISABLE_XET=1` and re-run to
fall back to classic LFS multipart.

`private` is deliberately not passed: it only applies when a repo is created, and this is
pushing to one that already exists.

In [ ]:
assert REPO_ID != PLACEHOLDER, (
    "Set REPO_ID in the config cell, or export HF_DATASET_REPO, before pushing."
)
assert os.environ.get("HF_TOKEN"), (
    "HF_TOKEN is not set. Export it in the shell and restart the kernel -- otherwise this "
    "fails with a 401 having uploaded nothing."
)

ds.push_to_hub(
    REPO_ID,
    split=SPLIT,
    max_shard_size=MAX_SHARD_SIZE,
    commit_message=f"Add ClinicalTrials.gov corpus ({ds.num_rows:,} trials)",
)

## 6. Verify

Streaming, so this reads one shard header instead of pulling 2.2 GB back down.

In [ ]:
from datasets import load_dataset

remote = load_dataset(REPO_ID, split=SPLIT, streaming=True)
first = next(iter(remote))

print(f"columns match local : {sorted(first) == sorted(ds.column_names)}")
print(f"first nct_id        : {first.get('nct_id')}")
print(f"https://huggingface.co/datasets/{REPO_ID}")